In [1]:
import pandas as pd
import re
import sys, os
import yaml
with open('config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Plantes Indigenes 

In [2]:
df = pd.read_csv('data/local_df_ai_descriptors.csv')

In [3]:
# Extract unique edible parts, split into individual items, clean and normalize
unique_parts = df['Partie comestible'].dropna().unique()

edible_parts = []

for part in unique_parts:
    if not isinstance(part, str):
        continue
    
    # Split using both "," and ";" as possible delimiters
    for item in re.split(r'[;,]', part):
        cleaned = item.strip().lower()
        if cleaned:
            edible_parts.append(cleaned)

# Convert to a unique set
edible_parts = set(edible_parts)


In [4]:
lexicon = {'graines de sirop': 'additive',
 'bulbes': 'vegetable-root',
 'sève des pousses': 'additive',
 'fleurs': 'flower',
 'condiment': 'herb',
 'écorce': 'vegetable-stem',
 'microorganismes': 'animalproduct',
 'résine': 'spice',
 'bases des feuilles': 'vegetable-stem',
 'branches': 'vegetable-stem',
 'tiges': 'vegetable-stem',
 'thé': 'beverage',
 'graines': 'nutseed-seed',
 'thalle': 'plant',
 'sève': 'additive',
 'porte-greffes': 'plant',
 'legumes verts': 'Vegetable',
 'latex': 'additive',
 'aiguilles': 'herb',
 'légumes verts frais': 'vegetable',
 'fruits': 'fruit',
 'sucre': 'additive',
 'aucune partie': 'dish',
 'brindilles': 'herb',
 'racine pivotante': 'vegetable-root',
 'rhizomes': 'vegetable-root',
 'gomme': 'additive',
 'jeunes graines': 'nutseed-seed',
 'pousses': 'plant',
 'extrémités des branches': 'vegetable-stem',
 'grains': 'cereal',
 'cambium': 'plant',
 'feuilles': 'plant',
 'jeunes pousses': 'plant',
 'légumes verts': 'vegetable',
 'champignon': 'fungus',
 'ecorce intérieure': 'plant',
 'écorce interne': 'plantd',
 'oreilles intérieures': 'vegetable',
 'graines pour substitut de café': 'beverage',
 'jeunes pousses stériles': 'plant',
 'frondes': 'plant',
 'porte-greffe': 'plant',
 'tubercules': 'vegetable-tuber',
 'écorce intérieure': 'plant',
 'pollen': 'spice',
 'graines': 'nutseed-seed',
 'boutons floraux et jeunes fruits': 'flower',
 'ecorce interne': 'plant',
 'boutons floraux': 'flower',
 'racines': 'vegetable-root'}

In [5]:
import difflib

def find_best_match(x, lexicon):
  """
  Finds the best matching string in lexicon.keys() for string x.
  """
  matches = difflib.get_close_matches(x, lexicon.keys(), n=1, cutoff=0.6) # Adjust cutoff as needed
  if matches:
    return matches[0]
  else:
    return None

df['sub_category'] = ""
df = df.reset_index(drop=True)

for idx in df.index.to_list():
  cat = df.iloc[idx]['Partie comestible']
  if not isinstance(cat, str):
    continue

  elif "," in cat:
      cat = cat.split(",")[0]

  elif ";" in cat:
      cat = cat.split(";")[0]

  new_cat = find_best_match(cat, lexicon)
  new_cat = lexicon[new_cat]
  df.at[idx, 'sub_category'] = new_cat

# Cases I can't seem to catch
df.loc[df['sub_category']=='vegetable-tuber', 'sub_category'] = 'vegetable'
df.loc[df['sub_category']=='plantd', 'sub_category'] = 'plant'
df.loc[df['sub_category']=='vegetable-stem', 'sub_category'] = 'plant'
df.loc[df['sub_category']=='animalproduct', 'sub_category'] = 'animal product'

df['sub_category'] = df['sub_category'].str.lower()

df['sub_category'].value_counts()

sub_category
plant             68
vegetable-root    43
fungus            14
vegetable         11
fruit              9
additive           7
herb               6
flower             3
                   2
cereal             2
animal product     1
beverage           1
dish               1
Name: count, dtype: int64

# FlavorDB

In [6]:
flavor_db = pd.read_csv('data/flavor_db_descriptors.csv')
flavor_db['sub_category'] = flavor_db['category_readable'].str.lower().copy()
flavor_db.head()

,category,entity_id,category_readable,entity_alias_basket,entity_alias_readable,natural_source_name,entity_alias,molecules,natural_source_url,entity_alias_url,entity_alias_synonyms,error,sub_category,descriptors,common_name,answer,sys_prompt,user_prompt
0,animal product,0,Animal Product,"egg, egg-boiled, egg-cooked, egg-scrambled",Egg,Chicken,egg,"[{'bond_stereo_count': 0, 'undefined_atom_ster...",https://en.wikipedia.org/wiki/Chicken,https://en.wikipedia.org/wiki/Egg_as_food,Egg,NaN,animal product,"sulfurous, creamy, rich, mild, savory, buttery...",Egg,"mild, savory, rich, creamy, umami, slightly su...",You are a food science expert. Your task is to...,Ingredient:\nCommon name: Egg\nReturn a comma-...
1,prepared food,1,Bakery,bakery-products,Bakery Products,Poacceae,bakery,"[{'bond_stereo_count': 0, 'undefined_atom_ster...",https://en.wikipedia.org/wiki/Poaceae,https://en.wikipedia.org/wiki/Bakery,Bakery Products,NaN,bakery,Sure! Here is a comma-separated list of flavor...,Bakery Products,"sweet, yeasty, buttery, bready, toasted, nutty...",You are a food science expert. Your task is to...,Ingredient:\nCommon name: Bakery Products\nRet...
2,prepared food,2,Bakery,"bread, bread-preferment",Bread,Poacceae,bread,"[{'bond_stereo_count': 0, 'undefined_atom_ster...",https://en.wikipedia.org/wiki/Poaceae,https://en.wikipedia.org/wiki/Bread,Bread,NaN,bakery,"yeasty, toasty, nutty, malty, sweet, earthy, w...",Bread,"mildly sweet, yeasty, wheaty, toasty, savory, ...",You are a food science expert. Your task is to...,Ingredient:\nCommon name: Bread\nReturn a comm...
3,prepared food,3,Bakery,bread-rye,Rye Bread,Rye,bread-rye,"[{'bond_stereo_count': 0, 'undefined_atom_ster...",https://en.wikipedia.org/wiki/Rye,https://en.wikipedia.org/wiki/Rye_bread,Rye Bread,NaN,bakery,"earthy, nutty, sour, tangy, malty, slightly sw...",Rye Bread,"earthy, tangy, slightly sour, nutty, malty, gr...",You are a food science expert. Your task is to...,Ingredient:\nCommon name: Rye Bread\nReturn a ...
4,prepared food,4,Bakery,bread-wheaten,Wheaten Bread,Wheat,bread-wheaten,"[{'bond_stereo_count': 0, 'undefined_atom_ster...",https://en.wikipedia.org/wiki/Wheat,https://en.wikipedia.org/wiki/Soda_bread,"Soda scones, Soda farls",NaN,bakery,"toasty, nutty, yeasty, malty, slightly sweet, ...",Wheaten Bread,"mild, wheaty, nutty, yeasty, slightly sweet, g...",You are a food science expert. Your task is to...,Ingredient:\nCommon name: Wheaten Bread\nRetur...


# Harmonizing categories

In [7]:
cat_mapping = {
    "plant": "plant",

    "plantderivative": "plant",
    "flower": "plant",
    "herb": "plant",
    "spice": "plant",
    "essentialoil": "plant",
    "seed": "nuts, legumes",
    "additive": "plant",
    "fruit": "fruit",
    "fruit-berry": "fruit",
    "berry": "fruit",
    "fruit-citrus": "fruit",
    "fruit-essence": "fruit",
    "vegetable": "vegetable",
    "vegetable": "vegetable",
    "vegetable-tuber": "tuber",
    "vegetable-root": "tuber",
    "vegetable-stem": "vegetable",
    "vegetable-cabbage": "vegetable",
    "vegetable-fruit": "vegetable",
    "vegetable-gourd": "vegetable",
    "cereal": "grains",
    "cerealcrop-cereal": "grains",
    "cerealcrop-maize": "grains",
    "nutseed-nut": "nuts, legumes",
    "nutseed-seed": "nuts, legumes",
    "nutseed-legume": "nuts, legumes",
    "animalproduct": "animal product",
    "meat": "animal product",
    "fishseafood-fish": "animal product",
    "fishseafood-seafood": "animal product",
    "dairy": "animal product",
    "fungus": "fungus",
    "beverage": "beverage",
    "beverage-alcoholic": "beverage",
    "beverage-caffeinated": "beverage",
    "dish": "prepared food",
    "bakery": "prepared food",
    None: "other / unknown"
}



In [8]:
df['db'] = 'local'

flavor_db['db'] = 'flavor_db'
flavor_db.rename(columns={'entity_alias_readable': 'Nom scientifique', 'descriptors': "ai_descriptors"}, inplace=True)


merged_df = pd.concat([df[['Nom scientifique', 'ai_descriptors', 'sub_category', 'db']], flavor_db[['Nom scientifique', 'ai_descriptors','sub_category', 'db']]], ignore_index=True)

merged_df['category'] = merged_df['sub_category'].map(cat_mapping)
merged_df

,Nom scientifique,ai_descriptors,sub_category,db,category
0,Macrocystis pyrifera,"briny, umami, salty, oceanic, mineral, vegetal...",plant,local,plant
1,Palmaria palmata,"briny, salty, umami, oceanic, slightly sweet, ...",plant,local,plant
2,Porphyra perforata,"briny, umami, salty, mineral, oceanic, slightl...",plant,local,plant
3,Porphyra abbottae,"briny, umami, slightly sweet, mineral, oceanic...",plant,local,plant
4,Porphyra torta,"briny, oceanic, umami, mineral, slightly sweet...",plant,local,plant
...,...,...,...,...,...
1097,Green zucchini,"mild, fresh, slightly sweet, vegetal, watery, ...",vegetable,flavor_db,vegetable
1098,Yellow zucchini,"mild, sweet, slightly nutty, tender, delicate,...",vegetable,flavor_db,vegetable
1099,Saskatoon berry,"sweet, nutty, almond-like, earthy, fruity, ber...",berry,flavor_db,fruit
1100,Nanking cherry,"tart, sour, sweet, juicy, tangy, bright, acidi...",berry,flavor_db,fruit


In [9]:
merged_df['category'].value_counts()

category
plant             239
fruit             133
prepared food     110
animal product     94
tuber              43
vegetable          35
fungus             25
grains             25
beverage           14
nuts, legumes       5
Name: count, dtype: int64

In [10]:
merged_df.loc[merged_df['db'] =='local', 'category'].value_counts()

category
plant            84
tuber            43
fungus           14
vegetable        11
fruit             9
grains            2
beverage          1
prepared food     1
Name: count, dtype: int64

In [11]:
merged_df.to_csv('data/merged_ai_descriptors.csv', index=False)